# What do the constituent models rely on?

This is not a search for causal or universal quark/gluon features. It measures how trained
models respond to controlled input removal and which constituents locally influence a score.


## 1. Environment and compatible models


In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'sklearn', 'torch']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. From a terminal in this directory run "
        "./setup_student_env.sh, restart Jupyter with `henv . -x jupyter lab`, "
        "and select the Quark/Gluon Constituent ML kernel."
    )

import json, os, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import qg_constituent_ml as qg

DEVICE = qg.choose_device()
RUN_MODE = os.getenv('QG_RUN_MODE', 'quick')
SOURCE = Path(os.getenv('QG_INPUT_PATH', 'data/inclusive_jets.parquet'))
print(f'PyTorch {torch.__version__}; built with CUDA {torch.version.cuda}')
print(f'device={DEVICE}' + (f'; GPU={torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print(f'run mode={RUN_MODE}; source={SOURCE}')


In [ ]:
prepared=qg.prepare_dataset(SOURCE); manifest=qg.load_manifest(prepared)
bundles=qg.discover_bundles(dataset_fingerprint=manifest['source_sha256'])
if not bundles: raise FileNotFoundError('Train at least one architecture notebook first.')


## 2. Physics-aware group ablations

Zero in normalized space means replacing a continuous feature by its training mean. PID
ablation removes category information. Constituent removals preserve the remaining particles
but answer a model-reliance question—not a causal physics question.


In [ ]:
def predict_with_ablation(model,loader,kind):
    scores=[]; labels=[]; mean=np.array(manifest['normalization']['mean']); std=np.array(manifest['normalization']['std'])
    model.eval()
    with torch.no_grad():
      for b in loader:
        f=b['features'].to(DEVICE).clone(); c=b['coords'].to(DEVICE).clone(); m=b['mask'].to(DEVICE).clone()
        if kind=='momentum': f[:,:,0]=0
        elif kind=='angular': f[:,:,1:4]=0; c.zero_()
        elif kind=='pid': f[:,:,4:]=0
        else:
          raw_logz=f[:,:,0]*std[0]+mean[0]; z=torch.exp(raw_logz); dr=torch.linalg.vector_norm(c,dim=-1)
          if kind=='soft': m &= z>=0.05
          elif kind=='core': m &= dr>=0.10
          elif kind=='wide': m &= dr<0.20
        scores.append(torch.sigmoid(model(f,c,m)).cpu().numpy()); labels.append(b['labels'].numpy())
    return np.concatenate(labels).astype(int),np.concatenate(scores)

rows=[]
for bundle in bundles:
  model,config=qg.load_model_bundle(bundle,DEVICE); loader=qg.make_loaders(prepared,config['architecture'],config['mode'])[2]
  y,base=predict_with_ablation(model,loader,'none'); base_auc=qg.binary_metrics(y,base)['roc_auc']
  for kind in ('momentum','angular','pid','soft','core','wide'):
    _,score=predict_with_ablation(model,loader,kind); auc=qg.binary_metrics(y,score)['roc_auc']
    rows.append({'model':config['architecture'],'ablation':kind,'AUC':auc,'delta_AUC':auc-base_auc})
import pandas as pd
ablation=pd.DataFrame(rows); display(ablation.pivot(index='ablation',columns='model',values='delta_AUC'))


In [ ]:
pivot=ablation.pivot(index='ablation',columns='model',values='delta_AUC')
pivot.plot.bar(figsize=(10,4)); plt.axhline(0,color='black',lw=.8); plt.ylabel('AUC(ablation) - AUC(original)')
plt.title('Performance reliance on input groups'); plt.tight_layout(); plt.show()


## 3. Local integrated-gradient maps

For one jet, interpolate continuous inputs and angular coordinates from a mean/zero baseline.
PID stays fixed and should instead be studied by categorical occlusion. Attribution magnitude
is normalized within each model, so colors are not compared as absolute units across models.


In [ ]:
def integrated_gradient_map(model,batch,steps=24):
    full_f=batch['features'][:1].to(DEVICE); full_c=batch['coords'][:1].to(DEVICE); mask=batch['mask'][:1].to(DEVICE)
    base_f=full_f.clone(); base_f[:,:,:4]=0; base_c=torch.zeros_like(full_c); gf=torch.zeros_like(full_f); gc=torch.zeros_like(full_c)
    for alpha in torch.linspace(0,1,steps,device=DEVICE):
      f=(base_f+alpha*(full_f-base_f)).detach().requires_grad_(True); c=(base_c+alpha*(full_c-base_c)).detach().requires_grad_(True)
      score=torch.sigmoid(model(f,c,mask)).sum(); df,dc=torch.autograd.grad(score,(f,c),allow_unused=True)
      gf += torch.zeros_like(f) if df is None else df; gc += torch.zeros_like(c) if dc is None else dc
    attr=((full_f-base_f)*gf/steps).abs().sum(-1)+((full_c-base_c)*gc/steps).abs().sum(-1)
    return full_c[0].detach().cpu().numpy(),attr[0].detach().cpu().numpy(),mask[0].cpu().numpy()

fig,axes=plt.subplots(1,len(bundles),figsize=(5*len(bundles),4),squeeze=False)
for ax,bundle in zip(axes[0],bundles):
  model,config=qg.load_model_bundle(bundle,DEVICE); loader=qg.make_loaders(prepared,config['architecture'],config['mode'])[2]; batch=next(iter(loader))
  coords,attr,mask=integrated_gradient_map(model,batch); values=attr[mask]; values=values/(values.max()+1e-12)
  sc=ax.scatter(coords[mask,0],coords[mask,1],c=values,s=30+100*values,cmap='viridis'); ax.set(title=config['architecture'],xlabel=r'$\Delta\eta$',ylabel=r'$\Delta\phi$')
  fig.colorbar(sc,ax=ax,label='normalized attribution')
plt.tight_layout(); plt.show()


## 4. Interpretation limits

Large ablation losses identify reliance, not a uniquely important physical variable. Inputs
are correlated; removing them can create out-of-distribution jets. Generator truth labels are
idealized and process dependent. Attention weights and graph edges describe internal routing,
not causal explanations.
